# 1 — Boost PFC Basics

> **Goal.** Understand *why* power-factor correction exists, what
> PF and THD mean, and how the boost stage between a diode bridge
> and the output cap turns a "bad" rectifier (PF ≈ 0.6, THD > 100%)
> into a "good" one (PF > 0.99, THD < 5%). Preview the two strategies
> that the next notebooks dive into: **DCM** (natural PFC, single
> voltage loop) and **CCM** (active current shaping, two loops).

**Prerequisites**

- Boost converter modeling notebook (`projects/converters/boost/`).
  The PFC is a boost stage with rectified AC at the input, plus a
  control architecture that has to do TWO things at once (regulate
  $V_o$ AND shape $i_{in}$).

**What you'll know at the end**

1. Why a cheap rectifier-with-cap fails IEC 61000-3-2.
2. What "power factor" and "THD" actually mean.
3. How the boost topology after a bridge can fix both.
4. The DCM vs CCM trade-off: $L$ small or large, single loop or two
   loops, natural shaping or active shaping.
5. Where each mode lives in the design space.


## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from boost_pfc_model import (
    BoostPFCParams,
    dcm_steady_state_duty_exact,
    dcm_input_current_instantaneous,
    operating_point_report,
    power_factor, thd_current, line_band_filter,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 1. The "bad" rectifier

Imagine the simplest AC-to-DC front end: a diode bridge feeding a
large cap. The cap holds the output near $V_{g,pk}$, so the bridge
only conducts during the short interval each line cycle where
$|v_{ac}(t)|$ exceeds the cap voltage. The result is **narrow tall
current pulses** at twice the line frequency.

The line sees a current that's nothing like its voltage — full of
harmonics, mostly reactive. Power factor crashes.


In [ ]:
# Simulate a cheap bridge + bulk-cap "rectifier" to motivate PFC
f_line = 50.0
V_ac_pk = np.sqrt(2) * 230.0
t = np.linspace(0, 3.0/f_line, 6000)
v_ac = V_ac_pk * np.sin(2*np.pi*f_line*t)
v_g = np.abs(v_ac)

# Model: cap holds V_cap; diode conducts only when v_g > V_cap.
C_bulk = 470e-6
R_load = 1600.0  # 100 W at ~400 V
V_cap = V_ac_pk * 0.95  # start near peak
i_line = np.zeros_like(t)
v_cap_hist = np.zeros_like(t)
for k, tk in enumerate(t):
    dt = t[1] - t[0]
    if v_g[k] > V_cap:
        # Diode conducts; cap charged from v_g through a tiny series R
        I_load = V_cap / R_load
        # Crude pulsed-conduction model: enough to draw enough charge
        i_pulse = (v_g[k] - V_cap) / 1.0  # 1Ω equivalent series R
        i_pulse = max(i_pulse, 0.0)
        V_cap = V_cap + (i_pulse - I_load) * dt / C_bulk
        i_line[k] = i_pulse * np.sign(v_ac[k])
    else:
        I_load = V_cap / R_load
        V_cap = V_cap - I_load * dt / C_bulk
        i_line[k] = 0.0
    v_cap_hist[k] = V_cap

fig, axs = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax_top = axs[0]
ax_top.plot(t*1e3, v_ac, "C0", label="$v_{ac}(t)$")
ax_top.plot(t*1e3, v_cap_hist, "C2", label="$V_{cap}$ (bulk)")
ax_top.set_ylabel("Voltage [V]")
ax_top.legend(loc="lower right")
ax_top.set_title("Cheap diode-bridge + bulk-cap rectifier — the\\\"bad\\\" reference")

axs[1].plot(t*1e3, i_line, "C3", linewidth=1.0, label="$i_{line}(t)$")
axs[1].plot(t*1e3, V_ac_pk/np.max(np.abs(i_line))*i_line*0 + V_ac_pk/np.max(np.abs(v_ac))*v_ac, "C0:", alpha=0.4, label="$v_{ac}$ shape (scaled)")
axs[1].set_ylabel("Line current [A]")
axs[1].set_xlabel("Time [ms]")
axs[1].legend(loc="lower right")
plt.tight_layout(); plt.show()

# Measure PF on the steady-state portion
mask = t > 1.0/f_line
fs = 1.0/(t[1]-t[0])
pf_bad = power_factor(v_ac[mask], i_line[mask], f_s=fs, f_line=f_line)
i_filt = line_band_filter(i_line[mask], fs, f_line)
thd_bad = thd_current(i_filt, fs, f_line)
print(f"Bad rectifier — PF = {pf_bad:.3f}, THD = {thd_bad*100:.1f} %")
print(f"  (typical real designs measure PF ≈ 0.55-0.65, THD > 100 %)")


## 2. PF and THD — definitions

For periodic line-frequency signals:

$$
\text{PF} = \frac{\bar P}{V_{rms} I_{rms}}
= \underbrace{\cos(\varphi_1)}_{\text{disp. factor}}
\cdot
\underbrace{\frac{I_{1,rms}}{I_{rms}}}_{\text{distortion factor}}
$$

The displacement factor is the cosine of the fundamental phase
shift; the distortion factor is the ratio of fundamental RMS to
total RMS. For PFC stages we usually have $\varphi_1 \approx 0$
(current in phase with voltage) so:

$$
\text{PF} \approx \frac{1}{\sqrt{1 + \text{THD}^2}}
$$

$$
\text{THD} = \frac{\sqrt{\sum_{n \ge 2} I_n^2}}{I_1}
$$

So PF = 0.99 corresponds to THD ≈ 14%. PF = 0.95 ↔ THD ≈ 33%.

**Standard limits** (IEC 61000-3-2 Class D, applicable to PC and
consumer equipment > 75 W): 3rd harmonic ≤ 3.4 mA/W, 5th ≤ 1.9 mA/W,
etc. A boost PFC with PF > 0.95 passes these comfortably.


## 3. The boost PFC topology

Insert a boost converter between the diode bridge and the bulk cap.
The boost inductor forces a controlled current shape, decoupling
the **line-side** current from the **cap-side** charging behavior:

```
   v_ac ───┤                                            ┌─── V_o (400 V DC)
           │   bridge       ┌────┐    ┌── D ────┬───────┤
   v_ac ───┤              ──┤  L ├────┤         │       │
           │                └────┘    │        +C       R_load
           │                          S         │       │
           └──────────────────────────┴─────────┴───────┘
              |v_ac(t)| = v_g(t)        boost stage
```

The boost stage works exactly like a regular boost converter, **but
its input is no longer DC** — $v_g(t) = |v_{ac}(t)|$ is a 100-Hz
half-sine. The output bulk cap stores enough energy to ride out the
line zero-crossings (with a 2·$f_{line}$ ripple).

If we **control the inductor current to follow $|v_{ac}(t)|$**, the
line sees a resistive load — PF → 1, THD → 0.


## 4. Two strategies — DCM vs CCM

The boost stage can run in either conduction mode. The choice
fundamentally changes the controller:

| | DCM (small L) | CCM (large L) |
|---|---|---|
| $i_L$ per $T_{sw}$ | triangle → 0 | trapezoid above 0 |
| Avg $i_L$ shape | **automatic** $\propto v_g$ (if $D$ const) | needs an active current loop |
| Controller | **1 voltage loop** | **2 loops** (current + voltage) + multiplier |
| Plant for voltage loop | first-order (1/(1+sRC/2)) | first-order (after current loop closes) |
| Plant for current loop | — | integrator $V_o/(sL)$ |
| Peak $i_L$ | ~2× CCM | moderate |
| Has RHP zero? | **no** | yes (boost RHP zero) |

The headline: **DCM gives you PFC almost for free** — set $D$ ≈ const
over a line half-cycle, the natural physics shapes the current. Pay
in higher peak current and worse EMI.

**CCM gives lower stress and cleaner waveforms** at the cost of a
significantly more complex controller (and bringing back the boost's
RHP zero in the current loop dynamics).


In [ ]:
# Show the two operating points side by side
p_dcm = BoostPFCParams.dcm_design()
p_ccm = BoostPFCParams.ccm_design()
print("DCM design:")
print(operating_point_report(p_dcm, mode="DCM"))
print()
print("="*70)
print("CCM design:")
print(operating_point_report(p_ccm, mode="CCM"))


## 5. The inductor current — shape preview

Here's what $i_L$ actually looks like over a line cycle in each
mode. This is purely an analytical sketch (not yet a switched sim);
just the **envelope** of $i_L$ — the per-$T_{sw}$ peak or average.

- **DCM**: $i_L$ is a triangle each $T_{sw}$. Peak $i_{pk}(t) =
  v_g(t) \cdot D \cdot T_{sw}/L$ — proportional to $v_g(t)$.
  Returns to 0 each switching period (the "dis-continuous" in DCM).

- **CCM**: $i_L$ ripples around a non-zero average. With active
  current shaping, that average is forced to track $v_g(t)$:
  $\langle i_L \rangle(t) \propto v_g(t)$. The ripple is at $f_{sw}$,
  the envelope is at $f_{line}$.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

# DCM envelope at V_ac_nom = 230V
p = p_dcm
t_line = np.linspace(0, 1/p.f_line, 1000)
v_g = p.V_g_pk_nom * np.abs(np.sin(p.omega_line * t_line))
D = dcm_steady_state_duty_exact(p, V_ac=p.V_ac_nom)
# DCM peak: v_g · D · T_sw / L
i_pk = v_g * D * p.T_sw / p.L
# DCM average (cusp-distorted): v_g/Re · 1/(1-v_g/V_o)
Re = 2*p.L*p.f_sw/D**2
i_avg = v_g/Re / np.maximum(1.0 - v_g/p.V_o, 1e-3)
axs[0].plot(t_line*1000, i_pk, "C1", label="$i_{pk}(t)$ (DCM triangle peak)")
axs[0].plot(t_line*1000, i_avg, "C3", linewidth=2, label="$\\langle i_g \\rangle$ avg-per-$T_{sw}$")
axs[0].plot(t_line*1000, v_g / v_g.max() * i_pk.max() * 0.5, "C0:", alpha=0.5, label="$v_g$ shape")
axs[0].set_ylabel("Current [A]"); axs[0].set_xlabel("Time [ms]")
axs[0].set_title(f"DCM @ V_ac=230V, D={D:.3f}")
axs[0].legend(loc="lower right")

# CCM envelope: average i_L should be P_in/v_g instantaneously, but
# realistically it's P_in/V_ac_rms times a sin shape. Show the desired track.
p = p_ccm
v_g_c = p.V_g_pk_nom * np.abs(np.sin(p.omega_line * t_line))
# In CCM, controller forces <i_L>(t) = (P_in/V_ac_rms²) · v_g(t)
# scaling so that <v_g · i_L>_T = P_o
k_shape = p.P_o / (p.V_ac_nom ** 2)
i_avg_ccm = k_shape * v_g_c * 2  # ×2 because <sin²>=1/2
# Ripple amplitude (peak-to-peak)
delta_i_pp = v_g_c * (1.0 - v_g_c/p.V_o) * p.T_sw / p.L
i_top = i_avg_ccm + delta_i_pp/2
i_bot = np.maximum(i_avg_ccm - delta_i_pp/2, 0.0)
axs[1].fill_between(t_line*1000, i_bot, i_top, color="C1", alpha=0.3,
                    label="$i_L$ ripple envelope")
axs[1].plot(t_line*1000, i_avg_ccm, "C3", linewidth=2,
            label="$\\langle i_L \\rangle$ (current loop forces this)")
axs[1].plot(t_line*1000, v_g_c/v_g_c.max()*i_avg_ccm.max(), "C0:", alpha=0.5,
            label="$v_g$ shape")
axs[1].set_ylabel("Current [A]"); axs[1].set_xlabel("Time [ms]")
axs[1].set_title("CCM (with active current shaping)")
axs[1].legend(loc="lower right")

plt.tight_layout(); plt.show()


## 6. The voltage-loop bandwidth ceiling

Both modes have the **same** outer-loop constraint: the voltage
compensator must be MUCH slower than the line. Why?

The output voltage has a $2 f_{line}$ ripple (100 Hz at 50 Hz line),
because the boost stage delivers power in a pulsed pattern with
peaks near $|v_{ac}|$ maxima. If the voltage loop is fast enough to
"see" that ripple, it will respond by chopping the current reference
at 2·$f_{line}$ — which **distorts the input current shape** and
crashes PF.

**Rule of thumb**: $f_{c,v} \le 2 f_{line} / 10$.

At 50 Hz: $f_{c,v}$ ≤ 10 Hz. The voltage loop is **slow** —
settling times measured in 100s of ms, not in µs like the DC-DC
converters we've seen.

This makes PFC compensators feel different from DC-DC compensators:
- the plant is **easy** (first-order in both DCM and CCM)
- the bandwidth is **constrained** to be slow (not by stability,
  but by signal integrity)
- a simple **PI** is usually enough — Type-III is overkill.


## 7. Summary and what's next

| | DCM PFC | CCM PFC |
|---|---|---|
| $L$ | 100 µH (small) | 1 mH (large) |
| Voltage-loop plant | $K/(1+sRC/2)$ | $K/(1+sRC/2)$ |
| Current loop | **none** | $V_o/(sL)$ — integrator |
| Multiplier | no | yes |
| RHP zero | **no** | yes (in current-loop transient) |
| Best for | $\le$ 150 W | $\ge$ 200 W |

**Next notebooks**:

- `02_boost_pfc_dcm.ipynb` — analyze DCM in detail. Derive the
  equivalent resistance $R_e$, the cusp-distortion factor
  $F(m_{pk})$, the first-order voltage-loop plant. Design a PI.
  Run the switched closed-loop sim and verify PF/THD.

- `03_boost_pfc_ccm.ipynb` — analyze CCM. Two loops: inner current
  loop (PI on an integrator plant) + outer voltage loop (PI on the
  same first-order plant as DCM). Multiplier between them.
  Switched closed-loop sim.

- `04_boost_pfc_simulation.ipynb` — DCM vs CCM side by side.
  Line-step (90 → 230 V), load-step (full → half load), PF/THD
  across the line range.
